# Part 3 — Adaptive low-rank adaptation (LoRA / SoRA) for **pretrained Mamba**

Backbone: **`state-spaces/mamba-130m-hf`** (a genuinely pretrained Mamba), loaded
through HuggingFace `transformers` with `use_mambapy=False` so it runs on Colab via
the pure-PyTorch **slow path** — no CUDA `mamba_ssm` kernels required. We attach a
classification head, freeze the backbone, and fine-tune with LoRA and SoRA on
**CoLA**, reporting the **same four metrics as Part 1** (MCC, trainable params,
effective rank, time).

**Adapter targets inside each Mamba block** (same conceptual choice as the writeup):
`in_proj`, `out_proj` (static channel mixing) and **`x_proj`** — the projection that
produces the input-dependent selective-scan parameters Δ, B, C, i.e. Mamba's
*selectivity* mechanism. A single time-shared low-rank update is correct here because
the time-variation comes from the *inputs* to `x_proj`, not its weights.

> This is real PEFT: a pretrained backbone is frozen and only ~0.1–1% of parameters
> (the adapters + head) are trained, so CoLA MCC is meaningful (unlike a from-scratch
> backbone).

In [ ]:
!pip install -q transformers datasets scikit-learn pandas

In [ ]:
# === CoLA data + pretrained Mamba tokenizer =================================
import os, csv, urllib.request, zipfile, torch
from transformers import AutoTokenizer

if not os.path.exists("glue_data/CoLA/train.tsv"):
    os.makedirs("glue_data", exist_ok=True)
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/glue/data/CoLA.zip", "glue_data/CoLA.zip")
    with zipfile.ZipFile("glue_data/CoLA.zip") as z: z.extractall("glue_data/")

def read_cola(p):
    s, y = [], []
    with open(p, encoding="utf-8") as f:
        for row in csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE):
            if len(row) >= 4: y.append(int(row[1])); s.append(row[3])
    return s, y
tr_s, tr_y = read_cola("glue_data/CoLA/train.tsv")
va_s, va_y = read_cola("glue_data/CoLA/dev.tsv")
print(f"train={len(tr_s)} val={len(va_s)}")

MODEL = "state-spaces/mamba-130m-hf"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token

MAXLEN = 64
def encode(sents):
    e = tok(sents, padding="max_length", truncation=True, max_length=MAXLEN, return_tensors="pt")
    return e.input_ids, e.attention_mask.sum(1)          # ids, real-token lengths
Xtr, Ltr = encode(tr_s); Ytr = torch.tensor(tr_y)
Xva, Lva = encode(va_s); Yva = torch.tensor(va_y)
print("encoded:", Xtr.shape)

In [ ]:
# === Adapters (SoRA / LoRA) + Mamba classifier wrapper ======================
import torch.nn as nn, torch.nn.functional as F, time
from sklearn.metrics import matthews_corrcoef

class SoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.0):
        super().__init__(); self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.r, self.scaling = r, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        self.gate   = nn.Parameter(torch.ones(r)); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + self.scaling * F.linear(F.linear(self.drop(x), self.lora_A) * self.gate, self.lora_B)
    @torch.no_grad()
    def proximal_step(self, thr):
        g = self.gate; g.copy_(torch.sign(g) * torch.clamp(g.abs() - thr, min=0.0))
    def effective_rank(self, eps=1e-6): return int((self.gate.abs() > eps).sum())

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.0):
        super().__init__(); self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.lora_A, a=5**0.5)
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r)); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + self.scaling * F.linear(F.linear(self.drop(x), self.lora_A), self.lora_B)

# in/out_proj (channel mixing) + x_proj (selective Δ,B,C). In the HF backend dt_proj
# is ALSO adaptable (called as a module); add it to TARGETS to include it.
TARGETS = {"in_proj", "out_proj", "x_proj"}

class MambaClassifier(nn.Module):
    def __init__(self, backbone, d_model, n_classes=2):
        super().__init__(); self.backbone = backbone
        self.norm = nn.LayerNorm(d_model); self.head = nn.Linear(d_model, n_classes)
    def forward(self, ids, lengths, labels=None):
        h = self.backbone(input_ids=ids).last_hidden_state       # (B,L,D)
        last = h[torch.arange(h.size(0), device=h.device), (lengths - 1).clamp(min=0)]  # last real token
        logits = self.head(self.norm(last))
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return logits, loss

def inject_adapter(model, Cls, r=8, alpha=16, dropout=0.1):
    for p in model.parameters(): p.requires_grad = False
    n = 0
    for mod in model.modules():
        for cn, ch in list(mod.named_children()):
            if isinstance(ch, nn.Linear) and cn in TARGETS:
                setattr(mod, cn, Cls(ch, r, alpha, dropout)); n += 1
    for nm, p in model.named_parameters():
        if nm.startswith(("head", "norm")): p.requires_grad = True
    return model, n

def count_trainable(m):
    return (sum(p.numel() for p in m.parameters() if p.requires_grad),
            sum(p.numel() for p in m.parameters()))
print("adapters + classifier ready; targets:", TARGETS)

In [ ]:
# === Build pretrained-backbone models =======================================
from transformers import MambaModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def fresh_backbone():
    # slow path (no CUDA kernels); downloads ~250MB once
    return MambaModel.from_pretrained(MODEL, use_mambapy=False)

D_MODEL = fresh_backbone().config.hidden_size   # 768 for mamba-130m
R = 8

m_lora, n = inject_adapter(MambaClassifier(fresh_backbone(), D_MODEL), LoRALinear, r=R)
print("LoRA adapters:", n, "| trainable %%: %.3f" % (100*count_trainable(m_lora)[0]/count_trainable(m_lora)[1]))
m_sora, n = inject_adapter(MambaClassifier(fresh_backbone(), D_MODEL), SoRALinear, r=R)
print("SoRA adapters:", n, "| trainable %%: %.3f" % (100*count_trainable(m_sora)[0]/count_trainable(m_sora)[1]))

In [ ]:
# === Train + evaluate (manual loop, MCC metric) =============================
EPOCHS, BS, LR = 3, 16, 5e-4
SORA_THR = 1e-3          # proximal soft-threshold per step; sweep up to prune harder

def train_eval(model, name, sora_thr=0.0):
    model = model.to(device)
    sora = [m for m in model.modules() if isinstance(m, SoRALinear)]
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    Xd, Ld, Yd = Xtr.to(device), Ltr.to(device), Ytr.to(device)
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train(); perm = torch.randperm(len(Xd))
        for i in range(0, len(Xd), BS):
            idx = perm[i:i+BS]
            _, loss = model(Xd[idx], Ld[idx], Yd[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            for m in sora: m.proximal_step(sora_thr)
        print(f"  [{name}] epoch {ep+1}/{EPOCHS} done")
    dt = time.time() - t0
    model.eval(); preds = []
    with torch.no_grad():
        for i in range(0, len(Xva), BS):
            preds.append(model(Xva[i:i+BS].to(device), Lva[i:i+BS].to(device))[0].argmax(-1).cpu())
    mcc = matthews_corrcoef(Yva.numpy(), torch.cat(preds).numpy())
    tr, tot = count_trainable(model)
    er = sum(m.effective_rank() for m in sora) if sora else None
    print(f"[{name}] MCC={mcc:.4f} | trainable={tr:,}/{tot:,} ({100*tr/tot:.3f}%) | eff_rank={er} | time={dt:.1f}s\n")
    return dict(method=name, mcc=round(mcc,4), trainable_params=tr,
                effective_rank=er, train_time_s=round(dt,1))

results = []
results.append(train_eval(m_lora, "LoRA"))
results.append(train_eval(m_sora, "SoRA", sora_thr=SORA_THR))

In [ ]:
# === Optional full fine-tuning reference (slower; flip to True) =============
RUN_FULL_FT = False
if RUN_FULL_FT:
    m_full = MambaClassifier(fresh_backbone(), D_MODEL)
    for p in m_full.parameters(): p.requires_grad = True
    results.insert(0, train_eval(m_full, "full-FT"))

In [ ]:
# === Comparison table (same 4 metrics as Part 1) ===========================
import pandas as pd
df = pd.DataFrame(results)
df.columns = ["Method", "CoLA (MCC)", "Trainable Params", "Effective Rank", "Train Time (s)"]
print(df.to_string(index=False))
df.to_csv("part3_mamba_pretrained_comparison.csv", index=False)